# 00 - Environment and Project Imports

هدف هذا الـ Notebook هو فحص البيئة وليس تنفيذ المشروع.

المطلوب في المقابلة أن نثبت أن الـ Notebook يعمل على ملفات المشروع نفسها، لذلك هنا نفحص:
- Python kernel
- مسار المشروع
- المكتبات الأساسية
- إمكانية استيراد Services من `src/`

In [1]:
from pathlib import Path
import sys, os, json, sqlite3, subprocess, time
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# يستطيع صديقك تغيير هذا المتغير إذا كانت artifacts خارج المشروع.
# مثال في PowerShell قبل فتح notebook:
# $env:IR_ARTIFACT_ROOT="E:\\ir_project_artifacts"
ARTIFACT_ROOT = Path(os.environ.get("IR_ARTIFACT_ROOT", r"E:\ir_project_artifacts"))


def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return Path(paths[0])

DB_PATH = first_existing([
    PROJECT_ROOT / "data" / "documents.sqlite",
    PROJECT_ROOT / "data" / "documents.db",
    ARTIFACT_ROOT / "documents.sqlite",
    ARTIFACT_ROOT / "documents.db",
])

TERRIER_INDEX_PATH = first_existing([
    PROJECT_ROOT / "indexes" / "terrier_medline",
    PROJECT_ROOT / "data" / "indexes" / "terrier_medline",
    ARTIFACT_ROOT / "indexes" / "terrier_medline",
])

BERT_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_bert",
    PROJECT_ROOT / "indexes" / "faiss_bert_full",
    PROJECT_ROOT / "data" / "rag_artifacts",
    ARTIFACT_ROOT / "indexes" / "faiss_bert_full",
    ARTIFACT_ROOT / "indexes" / "faiss_bert",
])

WORD2VEC_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_word2vec",
    PROJECT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec",
])

REPORTS_DIR = PROJECT_ROOT / "reports" / "evaluation_notebook"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT       =", PROJECT_ROOT)
print("ARTIFACT_ROOT      =", ARTIFACT_ROOT)
print("DB_PATH            =", DB_PATH, "exists=", DB_PATH.exists())
print("TERRIER_INDEX_PATH =", TERRIER_INDEX_PATH, "exists=", TERRIER_INDEX_PATH.exists())
print("BERT_INDEX_DIR     =", BERT_INDEX_DIR, "exists=", BERT_INDEX_DIR.exists())
print("WORD2VEC_INDEX_DIR =", WORD2VEC_INDEX_DIR, "exists=", WORD2VEC_INDEX_DIR.exists())

PROJECT_ROOT       = e:\in Desktop\IR_Project\IR
ARTIFACT_ROOT      = E:\ir_project_artifacts
DB_PATH            = E:\ir_project_artifacts\documents.sqlite exists= True
TERRIER_INDEX_PATH = E:\ir_project_artifacts\indexes\terrier_medline exists= True
BERT_INDEX_DIR     = E:\ir_project_artifacts\indexes\faiss_bert_full exists= True
WORD2VEC_INDEX_DIR = E:\ir_project_artifacts\indexes\faiss_word2vec_full exists= True


In [2]:
import platform
print("Python:", platform.python_version())
print("Executable:", sys.executable)

Python: 3.13.0
Executable: e:\in Desktop\IR_Project\IR\.venv\Scripts\python.exe


In [3]:
packages = [
    "numpy", "pandas", "sklearn", "matplotlib", "sqlite3",
    "ir_datasets", "sentence_transformers", "faiss", "gensim"
]

for package in packages:
    try:
        if package == "sklearn":
            import sklearn
        elif package == "sqlite3":
            import sqlite3
        elif package == "faiss":
            import faiss
        else:
            __import__(package)
        print("OK   ", package)
    except Exception as exc:
        print("ERROR", package, "=>", type(exc).__name__, exc)

OK    numpy
OK    pandas
OK    sklearn
OK    matplotlib
OK    sqlite3
OK    ir_datasets
OK    sentence_transformers
OK    faiss
OK    gensim


In [ ]:
modules = [
    "src.datasets.loader",
    "src.storage.document_store",
    "src.preprocessing.text_cleaner",
    "src.preprocessing.query_processor",
    "src.embeddings.bert_vectorizer",
    "src.embeddings.faiss_store",
    "src.retrieval.bm25_retriever",
    "src.retrieval.tfidf_retriever",
    "src.retrieval.bert_retriever",
    "src.retrieval.word2vec_retriever",
    "src.retrieval.parallel_hybrid_retriever",
    "src.retrieval.serial_hybrid_retriever",
    "src.query_refinement.query_refinement_service",
    "src.evaluation.evaluator",
    "src.evaluation.metrics",
]

for module in modules:
    try:
        __import__(module)
        print("OK import:", module)
    except Exception as exc:
        print("FAILED import:", module, "=>", type(exc).__name__, exc)

In [ ]:
def file_size_mb(path):
    path = Path(path)
    if not path.exists() or path.is_dir():
        return None
    return round(path.stat().st_size / (1024 * 1024), 2)


def artifact_status_df():
    rows = []
    checks = {
        "SQLite document store": DB_PATH,
        "Terrier index folder (TF-IDF/BM25)": TERRIER_INDEX_PATH,
        "BERT index folder": BERT_INDEX_DIR,
        "BERT FAISS index": BERT_INDEX_DIR / "index.faiss",
        "BERT doc_ids.pkl": BERT_INDEX_DIR / "doc_ids.pkl",
        "BERT doc_ids.jsonl": BERT_INDEX_DIR / "doc_ids.jsonl",
        "BERT metadata.json": BERT_INDEX_DIR / "metadata.json",
        "Word2Vec index folder": WORD2VEC_INDEX_DIR,
        "Word2Vec FAISS index": WORD2VEC_INDEX_DIR / "index.faiss",
        "Word2Vec model": WORD2VEC_INDEX_DIR / "word2vec.model",
        "Word2Vec doc_ids.pkl": WORD2VEC_INDEX_DIR / "doc_ids.pkl",
        "Word2Vec metadata.json": WORD2VEC_INDEX_DIR / "metadata.json",
    }
    for name, path in checks.items():
        rows.append({"artifact": name, "path": str(path), "exists": Path(path).exists(), "size_MB": file_size_mb(path)})
    return pd.DataFrame(rows)

artifact_status_df()